# Enkratna podloga in napadi nanjo

Cilji laboratorijske vaje so sledeči:
- spoznati postrojitev podatkov za potrebe šifriranja,
- spoznati delovanje enkratne podloge (OTP) (in posledično tokovnih šifer),
- spoznati tehniko napada na enkratno podlogo, ko se ključ uporabi večkrat,
- spoznati tehniko napada na enkratno podlogo, ko lahko napadalec spremeni tajnopis.

**POMEMBNO**

Namen te laboratorijske vaje se je seznaniti z načinom ravnanja s šiframi. Naša implementacija šifer bo osnovana na preprostosti in bo narejena **zgolj v pedagoške namene**. Šifra ni primerna za uporabo v praksi. 

Šifre, ki so za uporabo v praksi primerne in primerno implementirane, bomo spoznali kasneje.

Vaja je namenjena sprotnemu tedenskemu delu in učenju. Rešitve preverjajte s priloženimi testi ter s poskušanjem in opazovanjem rezultatov. Oddajte izpolnjeni `assignment.ipynb`; podatkovnih datotek ni treba oddajati.

## Ustvarjanje pravih naključnih vrednosti

Naključne vrednosti bomo v Pythonu ustvarili s pomočjo modula `secrets`. Za zdaj bo to najboljši približek pravim naključnim vrednostim; [povezava na dokumentacijo.](https://docs.python.org/3/library/secrets.html)

Najprej modul uvozimo, nato pa imlementiramo funkcijo, katera ustvari seznam naključnih bajtov za podano dolžino.

In [ ]:
import secrets

In [ ]:
def gen_key(length):
    """Vrne nakljucni niz bajtov podane dolzine"""
    return secrets.token_bytes(length)

## Naloga 1: Postroj podatkov

Vse moderne šifre delujejo na bitih (in bajtih). To pomeni, da moramo visokonivojske podatke (števila, nize, podatkovne strukture) pretvoriti v zaporedje ničel in enic, da jih lahko šifriramo.

Pri pretvorbi nizov uporabimo kodirno shemo. Najbolj pogosti možnosti sta ASCII ter UTF-8. 

Če uporabimo zgolj znake s kodirne tabele ASCII, dobimo pri obeh kodirnih shemah isti rezultat

In [ ]:
sporocilo1 = "dober dan"
sporocilo1_ascii = sporocilo1.encode("ascii")
sporocilo1_utf8 = sporocilo1.encode("utf8")

assert sporocilo1_ascii == sporocilo1_utf8

print(sporocilo1)
print(sporocilo1_ascii)
print(sporocilo1_utf8)
print(list(sporocilo1_ascii))

Če pa uporabimo tudi znake, ki jih na kodirni tabeli ASCII ni, denimo šumnike, pa pride do razlik.

In [ ]:
sporocilo2 = "dober večer"
sporocilo2_utf8 = sporocilo2.encode("utf8")
print(sporocilo2_utf8)

ASCII kodiranje v tem primeru ne deluje: pri kodiranju znaka `č` bo prišlo do napake.

In [ ]:
try:
    sporocilo2.encode("ascii")
except UnicodeEncodeError as error:
    print("Pričakovana napaka kodiranja ASCII:", error)
else:
    raise AssertionError("Pri kodiranju šumnikov z ASCII pričakujemo UnicodeEncodeError.")

Včasih želimo bajte predstaviti v šestnajstiškem zapisu. V ta namen uporabimo funkcijo `hex()`.

In [ ]:
print(f"Sporočilo '{sporocilo2}' predstavljeno kot:\nSeznam bajtov: {list(sporocilo2_utf8)}\nŠestnajstiški niz: {sporocilo2_utf8.hex()}")

Tako npr. prva dva znaka (`64`) predstavljata šestnajstiški zapis števila `100` ($100 = 6 \cdot 16^1 + 4 \cdot 16^0$), medtem ko vrednost `100` v ASCII/UTF-8 kodiranju predstavlja znak `d`

Funkcija, ki deluje v obratni smeri tj. ki iz seznama bajtov zgradi niz znakov, se imenuje `bytes.decode()`. 

In [ ]:
bajti = b'dober ve\xc4\x8der'
print(bajti.decode("utf8"))

Kodiranje celih števil je preprosteje: le uporabimo [funkcijo `int.to_bytes()`](https://docs.python.org/3/library/stdtypes.html#int.to_bytes) Funkciji moramo podati vsaj 2 argumenta: število bajtov, ki se naj uporabijo za kodiranje (tj. dolžina) in pravilo po katerem so bajti urejeni tj. ali gre za pravilo debelega ali tankega konca (`big` ali `little` endian). Opcijsko lahko nastavimo še, ali gre za predznačeno število.

In [ ]:
stevilo = 100
print(stevilo.to_bytes(1, 'big').hex())
print(stevilo.to_bytes(1, 'big'))

Včasih nas zanima, kateri številčni vrednosti (v kodiranju unikod) pripada posamezen znak. To izvemo s pomočjo funkcije `ord()`.

In [ ]:
for ch in "dober večer":
    print(ch, ord(ch))

Funkcija, ki deluje v obratno smer, tj. iz podane unikod vrednosti vrne pripadajoč znak, se imenuje `chr()`.

In [ ]:
for code in [100, 111, 98, 101, 114, 32, 118, 101, 269, 101, 114]:
    print(code, chr(code))

## Naloga 2: Implementacija ekskluzivne disjunkcije

Implementirajte metodo `xor_bytes(s1, s2)`, ki sprejme seznama bajtov, `s1` in `s2`, in vrne `bytes`, kjer so vrednosti posameznih bajtov izračunane kot rezultat operacije ekskluzivne disjunkcije nad posameznimi elementi seznama `s1` in `s2`.

Operacija ekskluzivne disjukncije v Pythonu je mogoča z uporabo znaka `^`:
```python
>>> 1 ^ 2
3
```

V sledeči celici lahko delovanje vaše funkcije preverite na treh enotskih testih. Če testi ne vrnejo napake, jih je vaša implementacija uspešno prestala.

Če se dolžini razlikujeta, obdelajte le skupno dolžino obeh vhodov.

In [ ]:
def xor_bytes(s1: bytes, s2: bytes) -> bytes:
    """Izvede operacijo XOR med podanima seznamoma bajtov in vrne seznam bajtov"""
    # VAŠA REŠITEV
    raise NotImplementedError()

xor_bytes([1, 16], [2, 1])

In [ ]:
assert xor_bytes([1, 2, 3], [3, 2, 1]) == bytes([2, 0, 2])
assert xor_bytes([0, 0, 0], [3, 2, 1]) == bytes([3, 2, 1])
assert xor_bytes([1, 1, 1], [1, 1, 1]) == bytes([0, 0, 0])

## Naloga 3: Šifra enkratne podloge (OTP)

Implementirajte šifrirni in dešifrirni algoritma enkratne podloge. Če uporabite rešitev iz naloge 2, bo implementacija trivialno kratka.

Ključ je lahko daljši od sporočila. Rezultat mora biti tipa `bytes` in imeti dolžino sporočila.

In [ ]:
def enc_otp(key: bytes, pt: bytes) -> bytes:
    """Sifrirni algoritem enkratne podloge"""
    # VAŠA REŠITEV
    raise NotImplementedError()

In [ ]:
assert enc_otp([1, 2, 3], [1, 1, 1]) == xor_bytes([1, 2, 3], [1, 1, 1])

In [ ]:
def dec_otp(key: bytes, ct: bytes) -> bytes:
    """Desifrirni algoritem enkratne podloge"""
    # VAŠA REŠITEV
    raise NotImplementedError()

In [ ]:
assert dec_otp([1, 2, 3], [1, 1, 1]) == xor_bytes([1, 2, 3], [1, 1, 1])

## Naloga 4: Šifriranje in dešifriranje

Uporabite metodi šifriranja in dešifriranja iz prejšnje naloge in zašifrirajte sporočilo `Enkratna podloga je popolno tajna šifra.` 

Implementacijo šifriranja podajte v telesu funkcije `example_enc()`. Za ključ uporabite vrednost `3d26ebcbc0b2ad0d15c6be1f6259fd89495451fc2245cd8dad40c480a87bbb3a7525a9ba4abb930417`. Pozor: podano vrednost ključa je potrebno prebrati kot seznam bajtov. To storite s pomočjo funkcije `bytes.fromhex(...)`.

In [ ]:
def example_enc() -> bytes:
    # VAŠA REŠITEV
    raise NotImplementedError()
    
example_enc()

In [ ]:
assert example_enc().hex() == "784880b9a1c6c36c35b6d17b0e369ae8693e34dc522abde2c12eaba0dc1ad15414056c1b23dde16539"

V telesu funkcije `example_dec()` dešifrirajte sporočilo `784880b9a1c6c36c35b6d17b0e369ae8693e34dc522abde2c12eaba0dc1ad15414056c1b23dde16539` in pri tem uporabite isti ključ kot ste ga pri šifriranju.

In [ ]:
def example_dec() -> bytes:
    # VAŠA REŠITEV
    raise NotImplementedError()
    
example_dec().decode("utf8")

In [ ]:
assert example_dec().decode("utf8") == "Enkratna podloga je popolno tajna šifra."

## Naloga 5: Napad na večkratno podlogo

Poglejmo, kaj lahko naredi napadalec, če isto podlogo uporabimo za šifriranje več sporočil.

### Naloga 5.1: Delno dešifriranje

Za začetek si pripravimo implementacijo dešifriranega algoritma, ki dešifrira tajnopis tudi, če nam kakšen del ključa manjka. Dešifrirani algoritem naj dešifrira kot običajno, edina izjema so znaki, ki je vrednost ključa enaka 0. V tem primeru, naj kot pripadajoč znak v čistopisu nastavi vrednost `*`.

In [ ]:
def dec_otp_partial(key: bytes, ct: bytes) -> bytes:
    """Desifrira samo tiste znake, kjer kljuc ni 0 -- ce je, kot znak nastavi simbol *"""
    # VAŠA REŠITEV
    raise NotImplementedError()

dec_otp_partial(
    bytes.fromhex("0000ebcbc0b2ad0d15c6be1f6259fd89495451fc0000cd8dad40c480a87bbb3a7525a9ba4abb930000"),
    bytes.fromhex("784880b9a1c6c36c35b6d17b0e369ae8693e34dc522abde2c12eaba0dc1ad15414056c1b23dde16539")
).decode("utf8")

In [ ]:
assert dec_otp_partial([1, 0, 1], [2, 2, 2]) == bytes([3, ord("*"), 3])
assert dec_otp_partial([1, 1, 1], [2, 2, 2]) == dec_otp([1, 1, 1], [2, 2, 2])

### Tajnopisi

Tajnopisi so shranjeni v binarni obliki v datotekah `data/ct_i.bin`. Preberimo jih z diska.

In [ ]:
def load_cts():
    cipher_texts = []
    for i in range(10):
        with open(f"data/ct_{i}.bin", "rb") as h:
            cipher_texts.append(h.read())
    return cipher_texts

cipher_texts = load_cts()

print("Izpis prvih 50 bajtov tajnopisa šestnajstiško")
for i, c in enumerate(cipher_texts):
    print(f"Tajnopis {i}, dolžina {len(c)}:  {c.hex()[:100]}...")

Opazimo, da so vsi tajnopisi dolgi točno 109 bajtov. 

Vaša naloga je, da 

1) ugotovite ključ in z njim 
2) dešifrirajte vsa sporočila. 

Pri tem lahko upoštevate še, da so sporočila v Slovenščini ter sestojijo le iz malih črk in presledkov: šumniki, števke in vsa ostala ločila so odstranjeni.

### Naloga 5.2: Funkcija `multiple_xor(pos, cts)`

Implementirajte funkcijo `multiple_xor(pos, cts)`, ki na vhodu prejme pozicijo znaka v tajnopisu in seznam tajnopisov. Metoda nato vrne bajt ključa oz. vrednost 0, če določi, da bajta v ključu na dani poziciji ni mogoče ugotoviti.

Namig: implementacija je dokaj kratka, če uporabite vgrajeno funkcijo `all`. V nasprotnem primeru boste potrebovali vgnezdeno zanko.

Za kandidata za presledek zahtevajte, da je XOR z vsakim drugim istoležnim bajtom enak 0 ali vsaj 65. Če najdete več kandidatov, uporabite prvega v podanem vrstnem redu; sicer vrnite 0.

In [ ]:
def multiple_xor(pos: int, cts: list[bytes]) -> int:
    # VAŠA REŠITEV
    raise NotImplementedError()

In [ ]:
def test_multiple_xor():
    """Testira implementacijo funkcije multiple_xor
    
    Če ne vrne napake, je implementacija pravilna"""
    
    test_key = bytes.fromhex("cd04c883d62ca6e58f3cc3c554bc77")
    test_cts = [
      enc_otp(test_key, "v ponedeljek in".encode("ascii")),
      enc_otp(test_key, "ob uri na mestu".encode("ascii")),
      enc_otp(test_key, "zelo nenavadno ".encode("ascii"))
    ]

    assert multiple_xor(0, test_cts) == 0
    assert multiple_xor(1, test_cts) == test_key[1]
    assert multiple_xor(2, test_cts) == test_key[2]
    assert multiple_xor(3, test_cts) == 0
    assert multiple_xor(4, test_cts) == test_key[4]
    assert multiple_xor(5, test_cts) == 0
    assert multiple_xor(6, test_cts) == test_key[6]
    assert multiple_xor(7, test_cts) == 0
    assert multiple_xor(8, test_cts) == 0
    assert multiple_xor(9, test_cts) == test_key[9]
    assert multiple_xor(10, test_cts) == 0
    assert multiple_xor(11, test_cts) == 0
    assert multiple_xor(12, test_cts) == test_key[12]
    assert multiple_xor(13, test_cts) == 0
    assert multiple_xor(14, test_cts) == test_key[14]
    
test_multiple_xor()

### Naloga 5.3: Funkcija `guess_key(cts)`

Implementirajte funkcijo `guess_key(cts)`, ki na vhodu prejme seznam tajnopisov in nato vrne seznam bajtov, ki predstavljajo ključ, s katerim so bili tajnopisi ustvarjeni. V primeri, da kakšen bajt v ključu ni mogoče ugotoviti, nastavite njegovo vrednost na 0.

Opomba: funkcija naj vrne podatek tipa `bytearray` in ne `bytes`. Razlika je v tem, da je podatkovni tip `bytes` nespremenljiv (angl. immutable), tip `bytearray` pa spremenljiv (angl. mutable). Slednje bo prišlo prav v nadaljevanju, ko bomo bajte v ključu ročno spreminjali.

In [ ]:
def guess_key(cipher_texts: list[bytes]) -> bytearray:
    # VAŠA REŠITEV
    raise NotImplementedError()

In [ ]:
gk = guess_key(cipher_texts)

def display_ciphertexts(key: bytes, cipher_texts: list[bytes]):
    for c in cipher_texts:
        print(dec_otp_partial(key, c).decode("ascii"))
        
display_ciphertexts(gk, cipher_texts)

### Naloga 5.4: Ugotavljanje manjkajočih bajtov v ključu

Nekatere bajte ključ je algoritem ugotovil, preostale pa poiščite sami. Nekaj namigov.

1. Če je v kakšnem čistopisu očitno, kaj naj bi znak `*` predstavljal, izvedite operacijo `xor` med tistim znakom in istoležnim znakom tajnopisa. Tako boste pridobili bajt ključa na tej poziciji. 
2. Trenutni ključ se nahaja v (globalni) spremenljivki `gk`, tajnopisi pa v seznamu `cipher_texts`. 
3. Ko spremenite posamezen bajt v ključu, lahko vse tajnopise izpišete s klicem funkcije `display_ciphertexts(gk, cipher_texts)`. 

Ključ ugotovite s poskušanjem in sklepanjem o vsebini dešifriranih sporočil, pri čemer si lahko pomagate z delnim ključem `gk`. Funkcija `complete_key(cipher_texts)` naj zgolj vrne končni, v celoti dopolnjeni ključ kot `bytearray`, v funkciji vam tako **ni treba implementirati algoritma za iskanje manjkajočih bajtov ključa:** ključ vanjo neposredno zapišite, na primer s klicem `return bytearray.fromhex("...")`.

**Pravilna rešitev:** vrnjeni ključ mora imeti dolžino 109 bajtov in pravilno dešifrirati vseh deset tajnopisov. Vrniti morate natančen ključ, ne zgolj besedila brez zvezdic.

In [ ]:
def complete_key(cipher_texts: list[bytes]) -> bytearray:
    """Vrne celoten 109-bajtni ključ za priložene tajnopise."""
    # VAŠA REŠITEV
    raise NotImplementedError()

In [ ]:
full_key = complete_key(cipher_texts)
display_ciphertexts(full_key, cipher_texts)

## Naloga 6: Gnetljivost tajnopisa

Pri zadnji nalogi bomo kot napadalec spremenili tajnopis, tako da bo sprememba vidna v čistopisu. Zgodba je sledeča.

Ana želi poslati zaupno pošto Boru, vas, ki igrate vlogo napadalca Nandija, pa vsebina sporočila _zares_ zanima.

Na srečo vam gre nekaj reči na roko. 

Prvič, Anin računalnik nima internetne povezave, vaš mobilni telefon pa. Zato ji prijazno ponudite, da zanjo postavite mobilno dostopno točko, preko katere se bo lahko povezala v internet in dostopala do poštnega strežnika. Poštni strežnik bo nato sporočilo dostavil Boru. 

Ana in Bor uporabljata poseben protokol za pošto: protokol FMTP -- Funny Mail Transfer Protocol.  Gre za preprost besedilno-osnovan protokol: prva vrstica označuje naslov prejemnika, druga naslov pošiljatelja, tretja zadevo, četrta je vedno prazna in na koncu je telo sporočila.

Vse, kar mora Anin poštni odjemalec storiti, da pošlje sporočilo, je, da strežniku FMTP dostavi besedilni niz, podoben naslednjemu.

```txt
prejemnik@enadomena.com
posiljatelj@drugadomena.com
Zadeva sporocila

<Telo sporocila>
```

Vse morebitne predhodne ali zaključne presledke v vsaki vrstici poštni strežnik ignorira oz. odstrani pred obdelavo. Na primer, e-pošto zgoraj bi lahko napisali kot spodnjo in ne bi bilo nobene razlike.


```txt
                    prejemnik@enadomena.com               
    posiljatelj@drugadomena.com                    
Zadeva sporocila

<Telo sporocila>
```

Tretjič, Ana vam v dobri veri -- kakšna naivnost! -- pove, da pošilja elektronsko sporočilo Boru in da uporablja protokol FMTP. (Torej poznate strukturo sporočila in vsebino prve vrstice čistopisa.)

Četrtič, Ana uporablja enkratno podlogo, mehanizmov za zagotavljanje celovitosti pa ni.

Ker nastopate v vlogi **posrednika**, lahko sedaj spremenite sporočilo tako, da strežnik FMTP ne bo poslal pošte Boru, temveč jo bo poslal na naslov, ki ga nadzirate vi, npr. `nandi@obvlada.si`.

In [ ]:
# Ana in Bor si enkrat tedensko v zivo izmenjata 1000 bajtov
# nakljucnih vrednosti za morebitne potrebe sifriranja
# te vrednosti kot napadalec vi ne poznate in je pri nalogi ne smete uporabiti
ana_bor_psk = gen_key(1000)

In [ ]:
# Ana pripravi sporočilo
email = """bor@student.uni-lj.si
ana@student.uni-lj.si
Hej

Na faksu si pozabil kapo.""".strip()

# In ga šifrira z enkratno podlogo
ct = enc_otp(ana_bor_psk, email.encode("utf8"))

# Nato ga pošlje (to simuliramo) na streznik FMTP
print("Ana pošilja šifrirano sporočilo na strežnik")
print("Tajnopis (HEX):", ct.hex())

Sedaj je na potezi napadalec Nandi.

Implementirajte funkcijo `change_ct(ct, new_email)`, ki na vhodu prejme tajnopis in email naslov, vrne pa modificiran tajnopis. Spremenite tajnopis tako, da ga bo strežnik FMTP še vedno dešifriral, a kot prejemnik ne bo naveden Bor, temveč email naslov, ki je podan v argumentu `new_email`. Predpostavite lahko, da bo `new_email` vedno krajši ali enak od naslova `bor@student.uni-lj.si`.

Predpostavite, da je naslov sestavljen iz znakov ASCII. Vrnite `bytes` iste dolžine kot vhod; preostanka sporočila ne spreminjajte. Funkcija mora delovati samo s podanima argumentoma, brez uporabe ključa `ana_bor_psk` ali spremenljivke `email`.

In [ ]:
def change_ct(ct: bytes, new_email: str) -> bytes:
    # VAŠA REŠITEV
    raise NotImplementedError()

S poganjanjem spodnje celice lahko preverite, ali vaša funkcija deluje pravilno. 

In [ ]:
def fmtp_receive(key, ct):
    print("Streznik FMTP prejel sporocilo:", ct.hex())
    pt = dec_otp(key, ct)
    print("Desifrirano sporocilo:")
    print(pt.decode("utf8"))

fmtp_receive(ana_bor_psk, change_ct(ct, "nandi@obvlada.si"))